In [1]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os

# Cargar API key desde .env
load_dotenv()
NEWS_API_KEY = os.getenv("NEWS_API_KEY")

print(f"API key cargada: {'✓' if NEWS_API_KEY else '✗'}")

API key cargada: ✓


In [5]:
env_path = r"C:\Users\mauri\Documents\kronos-trading-system\.env"

with open(env_path, 'r') as f:
    contenido = f.read()
    
print("Contenido actual del .env:")
print(contenido)

Contenido actual del .env:
ï»¿OANDA_API_KEY=pendiente
FRED_API_KEY=pendiente
NEWS_API_KEY=pendiente



In [6]:
env_path = r"C:\Users\mauri\Documents\kronos-trading-system\.env"

# Escribe el .env correctamente sin BOM
with open(env_path, 'w', encoding='utf-8') as f:
    f.write("OANDA_API_KEY=pendiente\n")
    f.write("FRED_API_KEY=pendiente\n")
    f.write(f"NEWS_API_KEY=343909c0f48c40519a991f8f4baa83e7\n")

print("Guardado. Verifica el archivo.")

Guardado. Verifica el archivo.


In [7]:
from importlib import reload
import dotenv
reload(dotenv)
from dotenv import load_dotenv

load_dotenv(r"C:\Users\mauri\Documents\kronos-trading-system\.env", override=True)
NEWS_API_KEY = os.getenv("NEWS_API_KEY")

print(f"Longitud: {len(NEWS_API_KEY) if NEWS_API_KEY else 0}")

Longitud: 32


In [9]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os

load_dotenv(r"C:\Users\mauri\Documents\kronos-trading-system\.env", override=True)
NEWS_API_KEY = os.getenv("NEWS_API_KEY")
print(f"Listo. Key: {len(NEWS_API_KEY)} caracteres")

Listo. Key: 32 caracteres


In [10]:
def obtener_noticias_forex(query, dias_atras=7, api_key=NEWS_API_KEY):
    fecha_desde = (datetime.now() - timedelta(days=dias_atras)).strftime('%Y-%m-%d')
    
    url = "https://newsapi.org/v2/everything"
    params = {
        'q': query,
        'from': fecha_desde,
        'language': 'en',
        'sortBy': 'publishedAt',
        'pageSize': 20,
        'apiKey': api_key
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if data['status'] == 'ok':
        titulares = [a['title'] for a in data['articles'] if a['title']]
        return titulares
    else:
        print(f"Error: {data.get('message', 'desconocido')}")
        return []

queries = ["EUR USD forex", "ECB interest rates", "Federal Reserve dollar"]
todos_titulares = []

for query in queries:
    titulares = obtener_noticias_forex(query, dias_atras=7)
    todos_titulares.extend(titulares)
    print(f"'{query}': {len(titulares)} titulares")

print(f"\nTotal: {len(todos_titulares)}")
print("\nPrimeros 5:")
for t in todos_titulares[:5]:
    print(f"  - {t}")

'EUR USD forex': 3 titulares
'ECB interest rates': 19 titulares
'Federal Reserve dollar': 20 titulares

Total: 42

Primeros 5:
  - OneTwoMarkets Expands Multi-Asset Access as Demand Surges From Asian Retail Traders
  - DAILY CURRENT AFFAIRS IAS | UPSC Prelims and Mains Exam – 27th March 2026
  - xfinance added to PyPI
  - The Gulf war economic fallout: how bad is it and how long will it last?
  - Switching could save you over €10,000 if your mortgage fixed-rate is coming to an end


In [13]:
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

print("Cargando FinBERT...")
finbert = pipeline("text-classification", model="ProsusAI/finbert", return_all_scores=True)
print("Listo.")

def calcular_sentimiento_real(titulares):
    if not titulares:
        return 0.0, 0.0
    
    scores = []
    for titular in titulares:
        try:
            resultado = finbert(titular[:512])
            score_dict = {r['label']: r['score'] for r in resultado}
            score_neto = score_dict.get('positive', 0) - score_dict.get('negative', 0)
            scores.append(score_neto)
        except:
            continue
    
    if not scores:
        return 0.0, 0.0
    
    scores = np.array(scores)
    return round(float(np.mean(scores)), 4), round(float(np.std(scores)), 4)

# Procesar todos los titulares
print(f"\nProcesando {len(todos_titulares)} titulares con FinBERT...")
score_hoy, incertidumbre_hoy = calcular_sentimiento_real(todos_titulares)

print(f"\nSentimiento EUR/USD HOY ({datetime.now().strftime('%Y-%m-%d')}):")
print(f"  Score: {score_hoy} ({'Bullish' if score_hoy > 0.1 else 'Bearish' if score_hoy < -0.1 else 'Neutral'})")
print(f"  Incertidumbre: {incertidumbre_hoy}")

# Top 5 titulares más bearish y bullish
print("\nTop 3 titulares analizados:")
for titular in todos_titulares[:3]:
    resultado = finbert(titular[:512])
    score_dict = {r['label']: r['score'] for r in resultado}
    score = score_dict.get('positive', 0) - score_dict.get('negative', 0)
    sentimiento = 'BULLISH' if score > 0.1 else 'BEARISH' if score < -0.1 else 'NEUTRAL'
    print(f"  [{sentimiento} {score:.3f}] {titular[:70]}")

Cargando FinBERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Listo.

Procesando 42 titulares con FinBERT...

Sentimiento EUR/USD HOY (2026-04-04):
  Score: -0.2264 (Bearish)
  Incertidumbre: 0.5672

Top 3 titulares analizados:
  [BULLISH 0.931] OneTwoMarkets Expands Multi-Asset Access as Demand Surges From Asian R
  [NEUTRAL 0.000] DAILY CURRENT AFFAIRS IAS | UPSC Prelims and Mains Exam – 27th March 2
  [NEUTRAL 0.000] xfinance added to PyPI


In [14]:
# Guardar sentimiento real del día
registro_sentimiento = {
    'fecha': datetime.now().strftime('%Y-%m-%d'),
    'score': score_hoy,
    'incertidumbre': incertidumbre_hoy,
    'n_titulares': len(todos_titulares),
    'interpretacion': 'Bullish' if score_hoy > 0.1 else 'Bearish' if score_hoy < -0.1 else 'Neutral'
}

df_sentimiento = pd.DataFrame([registro_sentimiento])
df_sentimiento.to_csv("../data/processed/sentimiento_real_hoy.csv", index=False)

print("Sentimiento real guardado:")
print(df_sentimiento.to_string(index=False))

Sentimiento real guardado:
     fecha   score  incertidumbre  n_titulares interpretacion
2026-04-04 -0.2264         0.5672           42        Bearish
